In [16]:
total_cyclists = cyclist_profile['FE_PESS'].sum()
total_pnb_cyclists = pnb_cyclist_profile['FE_PESS'].sum()

zone_profile = {}

for zona in cyclist_profile['ZONA'].unique():
    zone_profile[zona] = {
        'total_cyclists': cyclist_profile[cyclist_profile['ZONA'] == zona]['FE_PESS'].sum(),
        'total_pnb_cyclists': pnb_cyclist_profile[pnb_cyclist_profile['ZONA'] == zona]['FE_PESS'].sum(),
        'sexo':
    }
     

,FE_PESS,ZONA,SEXO,IDADE,RAÇA,RENDA_FA,GRAU_INS,TIPO_DOM,DURACAO,DISTANCIA,PE_BICI,MOT_SRES,QT_AUTO,QT_MOTO
0,20.452542,1,1,67,4,7000.000000,4,1,30,1518.600013,7,2,1,0
93,20.805172,1,1,28,4,5779.081275,2,1,8,627.498207,1,3,1,0
350,36.085220,2,2,34,1,9433.288101,5,1,10,928.282823,7,3,1,0
354,89.778288,2,1,36,4,9433.288101,5,1,10,928.282823,7,3,1,0
371,222.072926,3,1,31,4,3000.000000,4,1,15,2220.216656,6,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75637,41.867395,341,2,48,1,8795.540000,5,1,15,878.080862,1,3,0,0
75639,19.614564,341,1,74,1,8795.540000,5,1,15,815.726670,7,7,0,0
75641,19.614564,341,2,73,1,8795.540000,5,1,15,815.726670,7,7,0,0
75836,166.642369,342,1,49,1,10000.000000,5,1,30,6261.804373,6,3,0,0


# PROBLEMAS

In [74]:
from shapely.validation import explain_validity
invalid_OD_zones_gdf = OD_zones_gdf_untreated[~OD_zones_gdf_untreated.is_valid]
explain_validity(invalid_OD_zones_gdf)

# Por muitos polígonos não são válidos, (declarado na célula 3) segue a lista de razões:

,geometry
8,Ring Self-intersection[334100.113609622 739806...
10,Self-intersection[335766.610853515 7397060.310...
11,Ring Self-intersection[335762.685647727 739705...
39,Ring Self-intersection[338658.671142381 739572...
40,Ring Self-intersection[338658.704127304 739572...
...,...
511,Ring Self-intersection[315652.949310819 740127...
512,Ring Self-intersection[304370.332846271 739657...
513,Ring Self-intersection[304370.332846271 739657...
517,Self-intersection[304438.957977715 7382590.458...


# CEMITÉRIO

In [27]:
import pandas as pd

def read_ibge_microdata(txt_path, layout_path, sheet_name='PESS'):
    # Read layout sheet
    layout = pd.read_excel(layout_path, sheet_name=sheet_name, engine='odf').iloc[1:]
    
    # Extract variable names and positions
    var_names = layout.iloc[:, 0].astype(str).tolist()
    starts = layout.iloc[:, 7].astype(float).astype(int).tolist()
    ends = layout.iloc[:, 8].astype(float).astype(int).tolist()
    
    # Build colspecs
    colspecs = [(start - 1, end) for start, end in zip(starts, ends)]
    
    # Read fixed-width file
    df = pd.read_fwf(txt_path, colspecs=colspecs, names=var_names)
    
    return df

txt_path = '.\\data\\Amostra_Pessoas_14munic.txt'
layout_path = '.\\Layout_microdados_Amostra.ods'

df = read_ibge_microdata(txt_path, layout_path)


In [46]:
dict_code_prof = {
    'V0601':{'sexo':{'1':'Masculino','2':'Feminino'}},
    'V0606':{'cor':{'1':'Branca','2':'Preta','3':'Amarela','4':'Parda','5':'Indigena','9':'Sem_declaracao'}},
    'V6400':{'nivel_de_instrucao':{'1':'Sem_instrucao','2':'fundamental_completo','3':'medio_completo','4':'superior_completo'}},
    'V0640':{'estado_civil':{'1':'casado','5':'solteiro','4':'viuvo','3':'divorciado','2':'desquitado'}},
    'V0645':{'trabalho':{'1':'um','2':'dois_ou_mais','':'nenhum'}},
    'V6526':'renda_em_salarios_minimos',
    'V0653':'horas_trabalhadas',
    'V0662':{'tempo_de_deslocamento_ao_trabalho':{'1':'<=5min','2':'6-30min','3':'31min-1h','4':'1-2h','5':'>2h'}},
    'V6940':{'categoria_profissional':{'1':'empregado_clt','2':'empregado_estatuario(militares_inclusos)','3':'empregado_sem_clt','4':'conta_propria','5':'empregador','6':'nao_remunerado','7':'trabalhador_subsistente'}},
    'V5080':'rendimento_familiar_per_capita_em_salarios_minimos',
    'V1005':{'situacao_do_setor':{'1':'area_urbanizada','2':'area_nao_urbanizada','3':'area_urbanizada_isolada','4':'area_rural_de_extensao_urbana','5':'aglomerado_rural','6':'aglomerado_rural','7':'aglomerado_rural','8':'area_rural_exclusive_aglomerado_rural'}},
    'V0221':{'motocicleta_para_uso_particular':{'1':'possui','2':'nao_possui'}},
    'V0222':{'automovel_para_uso_particular':{'1':'possui','2':'nao_possui'}},
}

# Separando os ciclistas gerais dos ciclistas vivendo na zona PNB

## eu preciso associar as linhas de pessoas com os setores do censo para perfilar cada zona - agora com os microdados. MAS os microdados estão desencontrados, os códigos de região não batem, preciso resolver isso.

In [6]:
sp_zones.sort_values('setor_cens')

,id,setor_cens,populacao,area_hect,habit_hect,ano_densid,an_censo,geometry
14905,30762,355030801000001,806.0,7.231360,111.458976,2010,2010,"POLYGON ((339816.876 7392593.185, 339762.085 7..."
17366,31092,355030801000002,913.0,7.180359,127.152410,2010,2010,"POLYGON ((339898.791 7392663.965, 339890.991 7..."
17336,31091,355030801000003,625.0,5.611048,111.387391,2010,2010,"POLYGON ((340010.027 7392443.461, 339999.011 7..."
14904,30761,355030801000004,572.0,6.856916,83.419422,2010,2010,"POLYGON ((340035.826 7392138.185, 339960.781 7..."
2738,25273,355030801000005,754.0,8.248083,91.415183,2010,2010,"POLYGON ((339655.092 7392153.506, 339647.532 7..."
...,...,...,...,...,...,...,...,...
126,13605,355030896000243,336.0,1.026497,327.326892,2010,2010,"POLYGON ((358263.25 7395587.106, 358263.354 73..."
127,13607,355030896000244,209.0,0.894462,233.660135,2010,2010,"POLYGON ((358308.049 7395452.215, 358307.823 7..."
5377,30584,355030896000245,339.0,1.110189,305.353396,2010,2010,"POLYGON ((354654.416 7398105.552, 354676.451 7..."
14789,30585,355030896000246,163.0,0.813345,200.406872,2010,2010,"POLYGON ((354778.438 7398059.051, 354812.268 7..."


In [16]:
census_microdata

,field_1,setor,peso_amostral,sexo,cor,nivel_de_instrucao,estado_civil,trabalho,renda_em_salarios_minimos,horas_trabalhadas,tempo_de_deslocamento_ao_trabalho,categoria_profissional,rendimento_familiar_per_capita_em_salarios_minimos,situacao_do_setor
0,611459,3550308005001,129311807439310,2,1,2,1.0,1.0,100000.0,40.0,1.0,5.0,83824.0,1
1,611460,3550308005001,129311807439310,1,4,2,1.0,1.0,235294.0,40.0,3.0,5.0,83824.0,1
2,611461,3550308005001,129311807439310,2,4,1,5.0,,,,,,83824.0,1
3,611462,3550308005001,129311807439310,1,4,1,,,,,,,83824.0,1
4,611463,3550308005001,105984161470355,2,1,1,5.0,1.0,39216.0,20.0,,,19608.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
552032,1163491,3550308005310,217929418261170,1,4,1,,,,,,,88758.0,1
552033,1163492,3550308005310,166223292329429,1,4,3,5.0,,,,,,132353.0,1
552034,1163493,3550308005310,166223292329429,1,4,3,5.0,1.0,137255.0,40.0,,5.0,132353.0,1
552035,1163494,3550308005310,166223292329429,2,4,4,5.0,1.0,392157.0,40.0,,3.0,132353.0,1


In [65]:
a = census_microdata['setor'].unique() 
b = sp_zones_dissolved['setor'].unique()
c = [item for item in a if item not in b]
print(len(a), len(b), len(c))

310 240 310


In [14]:
import geopandas as gpd

sp_zones = gpd.read_file('./data/sao_paulo_demographics.geojson')
sp_zones['setor_cens'] = sp_zones['setor_cens'].str[:-2]
sp_zones_dissolved = sp_zones.dissolve(by='setor_cens')
sp_zones_dissolved = sp_zones_dissolved.reset_index()
sp_zones_dissolved = sp_zones_dissolved.rename(columns={'setor_cens': 'setor'})
census_microdata = gpd.read_file('./data/biker_profile.csv')
census_microdata = census_microdata.merge(sp_zones_dissolved[['setor', 'geometry']], on='setor', how='left')



In [73]:
import geopandas as gpd
from shapely import union_all 
from shapely import make_valid 

OD_data_df = gpd.read_file('./data/od23_all.csv', encoding='latin_1')

OD_data_gdf = gpd.GeoDataFrame(
    OD_data_df, 
    geometry=gpd.points_from_xy(OD_data_df['CO_DOM_Y'], OD_data_df['CO_DOM_X']),
    crs="EPSG:4326"
)
OD_data_gdf = OD_data_gdf.to_crs(epsg=31983)

OD_zones_gdf_untreated = gpd.read_file('./data/Zonas_2023.shp')
OD_zones_gdf = make_valid(OD_zones_gdf_untreated)
OD_zones_gdf.set_crs(epsg=31983, inplace=True)



spsp_limits = gpd.read_file('./data/REGIAO5/SIRGAS_REGIAO5.shp')

OD_zones_spsp = OD_zones_gdf.clip(spsp_limits.union_all())

spsp_data = OD_data_gdf[OD_data_gdf.geometry.within(OD_zones_spsp.union_all())]

spsp_cyclist_data = spsp_data['MODOPRIN'] == '16'

spsp_OD_data_gdf = spsp_data[spsp_cyclist_data]
spsp_OD_data_gdf = spsp_OD_data_gdf.drop(index=spsp_OD_data_gdf.index[1::2])
# this dataframe contain the data of all cyclists living in Sao Paulo city
# in this particular dataset, those are all cyclists.

pnb_zone_gdf = gpd.read_file('./data/pnb_zone.shp')
pnb_zone = pnb_zone_gdf['geometry'][0]

spsp_pnb_OD = spsp_OD_data_gdf.geometry.apply(lambda p: pnb_zone.contains(p))
# long nonsense abreviation, sorry. It means the cyclists living in Sao Paulo's pnb zone

spsp_pnb_OD_data_gdf = spsp_OD_data_gdf[spsp_pnb_OD]
# this dataframe contains the cyclists living in the PNB zone of Sao Paulo city


### Teste: Há 'ciclistas secundários'?

In [3]:
secondary_cyclists = spsp_pnb_OD_data_gdf[(spsp_pnb_OD_data_gdf['MODOPRIN'].astype('int') != '16') &
                                          ((spsp_pnb_OD_data_gdf['MODO1'] == '16') |
                                           (spsp_pnb_OD_data_gdf['MODO2'] == '16') |
                                           (spsp_pnb_OD_data_gdf['MODO3'] == '16') |
                                           (spsp_pnb_OD_data_gdf['MODO4'] == '16')                                           
                                           )]
secondary_cyclists[['MODOPRIN', 'MODO1', 'MODO2', 'MODO3', 'MODO4']]
# why is row zero present? 

,MODOPRIN,MODO1,MODO2,MODO3,MODO4
0,16,16,0,0,0
1,16,16,0,0,0
93,16,16,0,0,0
94,16,16,0,0,0
371,16,16,0,0,0
...,...,...,...,...,...
75641,16,16,0,0,0
75835,16,16,0,0,0
75836,16,16,0,0,0
75858,16,16,0,0,0


## Em geral, não

Distância média bicicleta vs motocicleta (feito pois a maior razão para uso de bicicleta comparativamente é a curta distância com 63% dos ciclistas)

In [101]:
print(spsp_OD_data_gdf['DISTANCIA'].astype(float).mean(),
        '\n',
        spsp_OD_data_gdf['DISTANCIA'].astype(float).median())
print(motorcycle_users['DISTANCIA'].astype(float).mean(),
        '\n',
        motorcycle_users['DISTANCIA'].astype(float).median())

2570.9315821346063 
 1835.81180952733
5701.016768043678 
 2898.43492250559


PROPORÇÃO DE RAZÕES BICICLETA

In [80]:
# spsp_OD_data_gdf['FE_PESS'] = spsp_OD_data_gdf['FE_PESS'].astype(float)
spsp_OD_data_gdf.groupby(['PE_BICI'])['FE_PESS'].sum()/spsp_OD_data_gdf['FE_PESS'].sum()

PE_BICI
1    0.633141
2    0.101536
3    0.004633
4    0.047790
5    0.024874
6    0.024941
7    0.133728
9    0.029356
Name: FE_PESS, dtype: float64

PE_BICI

1    0.633141

2    0.101536

3    0.004633

4    0.047790

5    0.024874

6    0.024941

7    0.133728

9    0.029356


1 - Pequena distância

2 - Condução cara

3 - Ponto/Estação distante

4 - Condução demora para passar

5 - Viagem demorada

6 - Condução lotada

7 - Atividade física

8 - Medo de contágio

9 - Outros motivos

In [ ]:
spsp_OD_data_gdf['FE_PESS'] = spsp_OD_data_gdf['FE_PESS'].astype(float)
spsp_OD_data_gdf.groupby(['MOT_SRES'])['FE_PESS'].sum()/spsp_OD_data_gdf['FE_PESS'].sum()

MOT_SRES
1     0.141357
10    0.029263
2     0.174742
3     0.409019
4     0.161796
5     0.020949
6     0.014554
7     0.047857
9     0.000464
Name: FE_PESS, dtype: float64

MOT_SRES

1     0.141357

10    0.029263

2     0.174742

3     0.409019

4     0.161796

5     0.020949

6     0.014554

7     0.047857

9     0.000464


1 - Trabalho Indústria

2 - Trabalho Comércio

3 - Trabalho Serviços

4 - Escola/Educação

5 - Compras

6 - Médico/Dentista/Saúde

7 - Recreação/Visitas/Lazer

8 - Residência

9 - Procurar Emprego

10 - Assuntos Pessoais

11 - Refeição

In [ ]:
import pandas as pd
OD_data_df[OD_data_df['QT_MOTO'] > '0']
motorcycle_users = OD_data_df[OD_data_df['QT_MOTO'] > '0']
motorcycle_users = motorcycle_users.drop(index=motorcycle_users.index[1::2])
motorcycle_users['MOT_SRES'] = pd.to_numeric(motorcycle_users['MOT_SRES'], errors='coerce')
motorcycle_users['FE_PESS'] = motorcycle_users['FE_PESS'].astype(float) 
motorcycle_users.groupby(['MOT_SRES'])['FE_PESS'].sum()/motorcycle_users['FE_PESS'].sum()

MOT_SRES
1     0.079141
2     0.123689
3     0.325483
4     0.346196
5     0.025287
6     0.022886
7     0.028062
9     0.003397
10    0.038771
11    0.007086
Name: FE_PESS, dtype: float64

PROPORÇÃO DE RAZÕES MOTOCICLETA

MOT_SRES

1     0.079141 

2     0.123689 

3     0.325483

4     0.346196

5     0.025287

6     0.022886

7     0.028062

9     0.003397

10    0.038771

11    0.007086


1 - Trabalho Indústria

2 - Trabalho Comércio

3 - Trabalho Serviços

4 - Escola/Educação

5 - Compras

6 - Médico/Dentista/Saúde

7 - Recreação/Visitas/Lazer

8 - Residência

9 - Procurar Emprego

10 - Assuntos Pessoais

11 - Refeição

In [ ]:
cyclist_profile = spsp_OD_data_gdf[['FE_PESS', 
                                    'ZONA',
                                    'SEXO', 
                                    'IDADE', 
                                    'RAÇA', 
                                    'RENDA_FA', 
                                    'GRAU_INS', 
                                    'TIPO_DOM',
                                    'DURACAO',
                                    'DISTANCIA',
                                    'PE_BICI',
                                    'MOT_SRES',
                                    'QT_AUTO',
                                    'QT_MOTO'
                                    ]]

cyclist_profile = cyclist_profile.astype({
    'FE_PESS': 'float',
    'RENDA_FA': 'float',
    'DURACAO': 'int',
    'DISTANCIA': 'float',
    'QT_AUTO': 'int',
    'QT_MOTO': 'int'})

pnb_cyclist_profile = spsp_pnb_OD_data_gdf[['FE_PESS',
                                            'ZONA',
                                            'SEXO', 
                                            'IDADE', 
                                            'RAÇA', 
                                            'RENDA_FA', 
                                            'GRAU_INS', 
                                            'TIPO_DOM',
                                            'DURACAO',
                                            'DISTANCIA',
                                            'PE_BICI',
                                            'MOT_SRES',
                                            'QT_AUTO',
                                            'QT_MOTO'
                                            ]]

pnb_cyclist_profile = pnb_cyclist_profile.astype({
    'FE_PESS': 'float',
    'IDADE': 'int',
    'RENDA_FA': 'float',
    'DURACAO': 'int',
    'DISTANCIA': 'float',
    'QT_AUTO': 'int',
    'QT_MOTO': 'int'})
    
dict_code_prof = {
    'SEXO':{'1':'Masculino','2':'Feminino', '3':'Nao_Respondeu'},
    'RAÇA':{'1':'Branca','2':'Preta','3':'Amarela','4':'Parda','5':'Indigena','6':'Sem_declaracao'},
    'GRAU_INS':{'1':'Sem_instrucao','2':'fundamental1_completo','3':'fundamental2_completo','4':'medio_completo','5':'superior_completo'},
    'TIPO_DOM':{'1':'particular','2':'coletivo'},
    'MOT_SRES':{'1':'Trabalho_Industria','2':'Trabalho_Comercio','3':'Trabalho_Servicos','4':'Escola_Educacao','5':'Compras','6':'Saude','7':'Lazer','8':'Residencia','9':'Busca_Emprego','10':'Assuntos_Pessoais','11':'Refeicoes'},
}

for col, mapping in dict_code_prof.items():
    if isinstance(mapping, dict):
        cyclist_profile[col] = cyclist_profile[col].map(mapping)
        pnb_cyclist_profile[col] = pnb_cyclist_profile[col].map(mapping)

In [215]:
display(cyclist_profile.groupby(['GRAU_INS']).sum()['FE_PESS']/cyclist_profile['FE_PESS'].sum())


for col, mapping in dict_code_prof.items():
    if isinstance(mapping, dict):
        spsp_data[col] = spsp_data[col].map(mapping)
        # pnb_cyclist_profile[col] = pnb_cyclist_profile[col].map(mapping)
spsp_data['GRAU_INS']

GRAU_INS
Sem_instrucao            0.079711
fundamental1_completo    0.089786
fundamental2_completo    0.253730
medio_completo           0.334407
superior_completo        0.242365
Name: FE_PESS, dtype: float64

c:\Users\João Rahal\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
c:\Users\João Rahal\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
c:\Users\João Rahal\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_index

0         NaN
1         NaN
2         NaN
3         NaN
4         NaN
         ... 
102823    NaN
102824    NaN
102825    NaN
102826    NaN
102827    NaN
Name: GRAU_INS, Length: 76081, dtype: object

SEXO
Masculino    81.024096
Feminino     18.975904
Name: proportion, dtype: float64

SEXO
Feminino     0.221471
Masculino    0.778529
Name: FE_PESS, dtype: float64

QT_AUTO
0    0.580214
1    0.387356
2    0.032429
Name: FE_PESS, dtype: float64

QT_MOTO
0    0.919395
1    0.074065
2    0.006540
Name: FE_PESS, dtype: float64

QT_AUTO
0    0.560197
1    0.411349
2    0.028454
Name: FE_PESS, dtype: float64QT_MOTO
0    0.965628
1    0.034372
Name: FE_PESS, dtype: float64

In [166]:
cyclist_profile_options = (
    cyclist_profile[[
    'SEXO',
    'ZONA',
    'PE_BICI',
    'TIPO_DOM',
    'GRAU_INS',
    'RAÇA',
    'MOT_SRES',
    'FE_PESS'
    ]]
)

def get_value_counts(group):
    results = {}
    total_weight = group['FE_PESS'].sum()
    for col in cyclist_profile_options.columns:
        weighted_counts = group.groupby(col)['FE_PESS'].sum()
        results[col] = weighted_counts / total_weight
    return pd.Series(results)

cyclist_profile_options = cyclist_profile_options.groupby('ZONA').apply(get_value_counts)
cyclist_profile_options = cyclist_profile_options.drop(columns=['FE_PESS', 'ZONA'])

cyclist_profile_numerics = cyclist_profile[list(set(cyclist_profile.columns)-set(cyclist_profile_options.columns))]
cyclist_profile_numerics = cyclist_profile_numerics.groupby('ZONA').apply(lambda x: x.apply(lambda y: (y * x['FE_PESS']).sum() / x['FE_PESS'].sum() if y.name != 'FE_PESS' else y.sum()))
cyclist_profile_numerics = cyclist_profile_numerics.drop(columns=['FE_PESS'])

cyclist_profile = cyclist_profile_options.merge(cyclist_profile_numerics, on='ZONA', how='inner')

C:\Users\João Rahal\AppData\Local\Temp\ipykernel_14208\1684131956.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cyclist_profile_options = cyclist_profile_options.groupby('ZONA').apply(get_value_counts)


Agora que temos as zonas devidamente separadas, é uma questão de dispor isso nos mapas folium, para você poder analisar individualmente cada zona. Por enquanto eu vou fazer a geral, pois a amostragem é muito pequena, mas assim que conseguir consertar os microdados, mudarei para o censo.

### Análises:

1 - Sexo, idade, renda, duração, pe_bici, tipo-dom e grau-ins pra cada zona

2 - Geral qt_auto, qt_moto, mot_sres

3 - Apresentar a falta de microdados do censo

4 - Apresentar o PNB score atual e o com as rotas teorizadas

5 - Apresentar o PNB score por região

### Limpando o monstro que é o arquivo de pessoas (diminuí para 5% do tamanho original)

In [11]:
import pandas as pd

def read_ibge_microdata(txt_path, layout_path, sheet_name='PESS'):
    # Read layout sheet
    layout = pd.read_excel(layout_path, sheet_name=sheet_name, engine='odf').iloc[1:]
    
    # Extract variable names and positions
    var_names = layout.iloc[:, 0].astype(str).tolist()
    starts = layout.iloc[:, 7].astype(float).astype(int).tolist()
    ends = layout.iloc[:, 8].astype(float).astype(int).tolist()
    
    # Build colspecs
    colspecs = [(start - 1, end) for start, end in zip(starts, ends)]
    
    # Read fixed-width file
    df = pd.read_fwf(txt_path, colspecs=colspecs, names=var_names)
    
    return df

txt_path = '.\\data\\Amostra_Pessoas_35_RMSP.txt'
layout_path = '.\\Layout_microdados_Amostra.ods'

df = read_ibge_microdata(txt_path, layout_path)

dict_code_prof = {
    'V0010': 'peso_amostral',
    'V0011': 'setor',
    'V0601':{'sexo':{'1':'Masculino','2':'Feminino'}},
    'V0606':{'cor':{'1':'Branca','2':'Preta','3':'Amarela','4':'Parda','5':'Indigena','9':'Sem_declaracao'}},
    'V6400':{'nivel_de_instrucao':{'1':'Sem_instrucao','2':'fundamental_completo','3':'medio_completo','4':'superior_completo'}},
    'V0640':{'estado_civil':{'1':'casado','5':'solteiro','4':'viuvo','3':'divorciado','2':'desquitado'}},
    'V0645':{'trabalho':{'1':'um','2':'dois_ou_mais','':'nenhum'}},
    'V6526':'renda_em_salarios_minimos',
    'V0653':'horas_trabalhadas',
    'V0662':{'tempo_de_deslocamento_ao_trabalho':{'1':'<=5min','2':'6-30min','3':'31min-1h','4':'1-2h','5':'>2h'}},
    'V6940':{'categoria_profissional':{'1':'empregado_clt','2':'empregado_estatuario(militares_inclusos)','3':'empregado_sem_clt','4':'conta_propria','5':'empregador','6':'nao_remunerado','7':'trabalhador_subsistente'}},
    'V5080':'rendimento_familiar_per_capita_em_salarios_minimos',
    'V1005':{'situacao_do_setor':{'1':'area_urbanizada','2':'area_nao_urbanizada','3':'area_urbanizada_isolada','4':'area_rural_de_extensao_urbana','5':'aglomerado_rural','6':'aglomerado_rural','7':'aglomerado_rural','8':'area_rural_exclusive_aglomerado_rural'}},
    'V0221':{'motocicleta_para_uso_particular':{'1':'possui','2':'nao_possui'}},
    'V0222':{'automovel_para_uso_particular':{'1':'possui','2':'nao_possui'}},
}

unused_cols = [col for col in df.columns if col not in dict_code_prof.keys()]
df = df.drop(columns=unused_cols)

rename_map = {}

for key, value in dict_code_prof.items():
    if type(value) == str:
        rename_map[key] = value
    elif type(value) == dict:
        rename_map[key] = list(value.keys())[0]

df = df.rename(columns=rename_map)

cut_df = df[[col for col in df.columns if col in dict_code_prof.keys()]]
cut_df = df[(df['setor']>=3550308000000) & (df['setor']<=3550309000000)]		
cut_df.to_csv('biker_profile.csv')